In [1]:
import pandas as pd
import numpy as np

In [2]:
regular_season = pd.read_csv("./data/MRegularSeasonDetailedResults.csv")
tourney = pd.read_csv("./data/MNCAATourneyCompactResults.csv")
seeds = pd.read_csv("./data/MNCAATourneySeeds.csv")

In [3]:
# Prepare winning team stats
winners = regular_season[['Season','WTeamID','WScore','LScore','WFGM','WFGA','WTO','WOR','WDR']].copy()
winners.columns = ['Season','TeamID','Score','OppScore','FGM','FGA','TO','OR','DR']
winners['Win'] = 1

# Prepare losing team stats
losers = regular_season[['Season','LTeamID','LScore','WScore','LFGM','LFGA','LTO','LOR','LDR']].copy()
losers.columns = ['Season','TeamID','Score','OppScore','FGM','FGA','TO','OR','DR']
losers['Win'] = 0

# Combine
teams = pd.concat([winners, losers])

In [4]:
team_stats = teams.groupby(['Season','TeamID']).agg({
    'Score': 'mean',
    'OppScore': 'mean',
    'FGM': 'mean',
    'FGA': 'mean',
    'TO': 'mean',
    'OR': 'mean',
    'DR': 'mean',
    'Win': 'mean'
}).reset_index()

team_stats.rename(columns={
    'Score': 'avg_score',
    'OppScore': 'avg_opp_score',
    'Win': 'win_rate'
}, inplace=True)

In [5]:
# Clean seed (remove region letter)
seeds['Seed'] = seeds['Seed'].str[1:3].astype(int)

team_stats = team_stats.merge(
    seeds[['Season','TeamID','Seed']],
    on=['Season','TeamID'],
    how='left'
)

In [6]:
def create_matchups(tourney, team_stats):
    data = []

    for _, row in tourney.iterrows():
        season = row['Season']
        team1 = row['WTeamID']
        team2 = row['LTeamID']

        team1_stats = team_stats[(team_stats.Season == season) & (team_stats.TeamID == team1)]
        team2_stats = team_stats[(team_stats.Season == season) & (team_stats.TeamID == team2)]

        if team1_stats.empty or team2_stats.empty:
            continue

        team1_stats = team1_stats.iloc[0]
        team2_stats = team2_stats.iloc[0]

        features = {
            'Season': season,
            'Team1': team1,
            'Team2': team2,

            # Feature differences
            'win_rate_diff': team1_stats['win_rate'] - team2_stats['win_rate'],
            'score_diff': team1_stats['avg_score'] - team2_stats['avg_score'],
            'opp_score_diff': team1_stats['avg_opp_score'] - team2_stats['avg_opp_score'],
            'fgm_diff': team1_stats['FGM'] - team2_stats['FGM'],
            'fga_diff': team1_stats['FGA'] - team2_stats['FGA'],
            'to_diff': team1_stats['TO'] - team2_stats['TO'],
            'or_diff': team1_stats['OR'] - team2_stats['OR'],
            'dr_diff': team1_stats['DR'] - team2_stats['DR'],
            'seed_diff': team1_stats['Seed'] - team2_stats['Seed'],

            'label': 1
        }

        data.append(features)

        # Reverse (data augmentation)
        features_rev = features.copy()
        for key in list(features.keys()):
            if '_diff' in key:
                features_rev[key] = -features[key]

        features_rev['Team1'] = team2
        features_rev['Team2'] = team1
        features_rev['label'] = 0

        data.append(features_rev)

    return pd.DataFrame(data)

In [7]:
train_df = create_matchups(tourney, team_stats)

print(train_df.shape)
train_df.head()

(2898, 13)


,Season,Team1,Team2,win_rate_diff,score_diff,opp_score_diff,fgm_diff,fga_diff,to_diff,or_diff,dr_diff,seed_diff,label
0,2003,1421,1411,-0.151724,-1.593103,7.614943,-0.354023,1.526437,0.973563,-0.890805,-1.627586,0.0,1
1,2003,1411,1421,0.151724,1.593103,-7.614943,0.354023,-1.526437,-0.973563,0.890805,1.627586,-0.0,0
2,2003,1112,1436,0.237685,17.421182,7.112069,5.493842,9.852217,0.716749,2.213054,1.918719,-15.0,1
3,2003,1436,1112,-0.237685,-17.421182,-7.112069,-5.493842,-9.852217,-0.716749,-2.213054,-1.918719,15.0,0
4,2003,1113,1272,-0.172414,1.448276,3.344828,0.931034,-3.103448,0.206897,-0.379310,-2.655172,3.0,1


In [8]:
train_df.to_csv("train_input.csv", index=False)

In [17]:
stage1 = pd.read_csv("./data/SampleSubmissionStage1.csv")
stage2 = pd.read_csv("./data/SampleSubmissionStage2.csv")

In [18]:
def prepare_test_data(sample, team_stats):
    # Split ID
    sample[['Season','Team1','Team2']] = sample['ID'].str.split('_', expand=True).astype(int)

    # Merge Team1 stats
    df = sample.merge(
        team_stats,
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    )

    df = df.rename(columns=lambda x: x + "_1" if x not in ['ID','Season','Team1','Team2'] else x)

    # Merge Team2 stats
    df = df.merge(
        team_stats,
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    )

    df = df.rename(columns=lambda x: x + "_2" if x not in ['ID','Season','Team1','Team2'] and not x.endswith('_1') else x)

    # Create feature differences
    features = ['win_rate','avg_score','avg_opp_score','FGM','FGA','TO','OR','DR','Seed']

    test_df = pd.DataFrame()
    test_df['ID'] = df['ID']

    for f in features:
        test_df[f'{f}_diff'] = df[f'{f}_1'] - df[f'{f}_2']

    return test_df

In [19]:
test_stage1 = prepare_test_data(stage1, team_stats)
test_stage2 = prepare_test_data(stage2, team_stats)

In [20]:
test_stage1.to_csv("test_stage1_input.csv", index=False)
test_stage2.to_csv("test_stage2_input.csv", index=False)